# Memory from Scratch

**What you will build:** a small memory system that lets an assistant hold a long conversation without overflowing the context window. You already know the core trick (the list *is* the memory); now we manage that list.

Frameworks call these strategies **trimming** and **summarization** (PydanticAI's history processors, LangChain's message trimming, the "compaction" step in coding agents like Claude Code). One thing they are *not*: a **checkpointer**. A checkpointer (LangGraph, 2.3) *persists and resumes* conversation state across processes; what we build here decides *what to keep in the window*. The two are orthogonal, and mixing up the names causes real confusion later — you can persist an unbounded history, and you can trim an unpersisted one.

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/06_memory_from_scratch.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
# Setup — same as notebook 0.1. Run once.
!pip install -q openai

import os, getpass, json
from openai import OpenAI

if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass.getpass("Paste your OpenRouter API key: ")

MODEL = "meta-llama/llama-3.3-70b-instruct"
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

def ask(system, user, temperature=0):
    """One-shot helper: system + user in, text out."""
    r = client.chat.completions.create(
        model=MODEL, temperature=temperature,
        messages=[{"role":"system","content":system},{"role":"user","content":user}])
    return r.choices[0].message.content

print("Ready. Model:", MODEL)

## The core mechanic (recap) and its problem

From 0.1: memory is just re-sending previous messages. So the naive memory is "append everything forever":

```
turn 1:  [system, u1, a1]
turn 2:  [system, u1, a1, u2, a2]
turn 3:  [system, u1, a1, u2, a2, u3, a3]   ... grows without bound
```

Two problems appear as it grows: you eventually exceed the **context window**, and every turn gets **slower and more expensive** (you pay for the whole history every time). So real memory is about *choosing what to keep*.

## Strategy 1 — Sliding window (keep the last N turns)

The simplest bounded memory: always keep the system prompt, but only the most recent few exchanges. Cheap and predictable; the cost is that older details fall off the edge.

In [ ]:
class WindowMemory:
    def __init__(self, system, keep_turns=3):
        self.system = system
        self.keep = keep_turns          # number of recent user/assistant PAIRS to keep
        self.history = []               # list of {'role','content'}
        self.prompt_tokens = []         # per-turn input size — for the measurement below

    def add(self, role, content):
        self.history.append({"role": role, "content": content})

    def messages(self):
        recent = self.history[-(self.keep * 2 + 1):]     # last N pairs, plus the new user msg
        if recent and recent[0]["role"] == "assistant":  # never start the window mid-pair:
            recent = recent[1:]                          #   some providers reject histories that
        return [{"role": "system", "content": self.system}] + recent   # open with an assistant turn

    def say(self, user_text):
        self.add("user", user_text)
        r = client.chat.completions.create(model=MODEL, messages=self.messages(), temperature=0)
        self.prompt_tokens.append(r.usage.prompt_tokens)
        reply = r.choices[0].message.content
        self.add("assistant", reply)
        return reply

m = WindowMemory("You are a concise assistant.", keep_turns=2)
print(m.say("My favourite colour is teal."))
print(m.say("I have a dog named Pixel."))
print(m.say("I live in Bilbao."))
print(m.say("My sister's name is June."))
print(m.say("What colour did I mention?"))    # teal fell out of the window turns ago

print("\nwhat the model actually saw on that last turn:")
for msg in m.messages():
    print(f"  {msg['role']:9s} {msg['content'][:60]}")

Look at the printout of what the model *actually saw*: the teal fact is simply not in the window, so no amount of prompting can recover it. That's the trade-off in its rawest form — and printing the exact messages you sent is the first debugging move for any memory bug ("the model forgot X" almost always means "X wasn't in the window"). To keep old facts *without* keeping every word, we summarize.

## Strategy 2 — Summary memory (compress the old, keep the recent)

Keep a rolling **summary** of everything older, plus the last few verbatim turns. Older facts survive in compressed form; the context stays bounded. This is the strategy most production assistants use.

In [ ]:
class SummaryMemory:
    def __init__(self, system, keep_turns=2):
        self.system = system
        self.keep = keep_turns
        self.summary = ""              # rolling summary of older turns
        self.recent = []               # verbatim recent messages
        self.prompt_tokens = []        # per-turn input size — for the measurement below

    def _summarize(self, old_messages):
        text = "\n".join(f"{m['role']}: {m['content']}" for m in old_messages)
        return ask("Update the running summary. Keep all facts about the user. Be brief.",
                   f"Existing summary:\n{self.summary}\n\nNew turns:\n{text}")

    def messages(self):
        sys = self.system + (f"\n\nConversation summary so far:\n{self.summary}" if self.summary else "")
        return [{"role": "system", "content": sys}] + self.recent

    def say(self, user_text):
        self.recent.append({"role": "user", "content": user_text})
        r = client.chat.completions.create(model=MODEL, messages=self.messages(), temperature=0)
        self.prompt_tokens.append(r.usage.prompt_tokens)
        reply = r.choices[0].message.content
        self.recent.append({"role": "assistant", "content": reply})
        # if recent grew too big, fold the oldest pair into the summary
        if len(self.recent) > self.keep * 2:
            old, self.recent = self.recent[:2], self.recent[2:]
            self.summary = self._summarize(old)
        return reply

m = SummaryMemory("You are a concise assistant.", keep_turns=2)
print(m.say("My favourite colour is teal."))
print(m.say("I have a dog named Pixel."))
print(m.say("I live in Bilbao."))
print(m.say("What colour did I mention, and what's my dog's name?"))   # recalled via the summary
print("\n[running summary]:", m.summary)

This time the early facts survive — not because we kept every message, but because we compressed them into the summary. Memory is a budgeting problem. You decide what's worth spending context tokens on.

## The budget, measured

This whole notebook argues that memory is a token budget. Let's stop asserting and **measure**: the same six-turn conversation through three memories — naive (keep everything), window, summary — printing the *input* size of every call. `usage.prompt_tokens` (from 0.1) is the meter.

In [ ]:
class NaiveMemory:
    """Append everything forever — the baseline we're arguing against."""
    def __init__(self, system):
        self.msgs = [{"role": "system", "content": system}]
        self.prompt_tokens = []
    def say(self, user_text):
        self.msgs.append({"role": "user", "content": user_text})
        r = client.chat.completions.create(model=MODEL, messages=self.msgs, temperature=0)
        self.prompt_tokens.append(r.usage.prompt_tokens)
        reply = r.choices[0].message.content
        self.msgs.append({"role": "assistant", "content": reply})
        return reply

TURNS = [
    "My favourite colour is teal.",
    "I have a dog named Pixel.",
    "I live in Bilbao.",
    "My sister's name is June.",
    "I work as a marine biologist. Tell me one fun fact about my field.",
    "Summarize everything you know about me.",
]

naive  = NaiveMemory("You are a concise assistant.")
window = WindowMemory("You are a concise assistant.", keep_turns=2)
summ   = SummaryMemory("You are a concise assistant.", keep_turns=2)

for t in TURNS:
    naive.say(t); window.say(t); summ.say(t)

print("prompt tokens sent, per turn:")
print("turn   naive  window  summary")
for i in range(len(TURNS)):
    print(f"{i+1:4d}  {naive.prompt_tokens[i]:6d}  {window.prompt_tokens[i]:6d}  {summ.prompt_tokens[i]:7d}")

Read the columns: **naive grows every single turn** and never stops — extend the conversation to 50 turns and you're paying for (and slowing down on) thousands of tokens per call before you hit the context wall. **Window plateaus** at a constant size. **Summary plateaus** a bit higher (the summary text rides along in the system prompt) and additionally pays one summarizer call per folded turn — the cost of keeping old facts alive.

One column the table doesn't show: the *last* turn's answer. Ask each memory to "summarize everything you know about me" and naive knows all five facts, window knows the last two, summary knows whatever survived compression. Cost, capacity, fidelity — pick two. That's the trade the rest of this course keeps returning to.

::::{dropdown} 🔍 The two strategies, and where each fits
:color: info

```
Window   [system | ....... last N turns .......]        simple, forgets old facts
Summary  [system + summary | .. last N turns ..]        keeps old facts, costs a summarizer call
```

Real systems combine both, and add a third: **retrieval** — instead of summarizing, store old turns in a database and fetch only the relevant ones on demand. That's RAG, and it's the same trick applied to documents. You'll build it as the knowledge agent in **1.7**.

**On counting tokens:** to budget precisely you need to *count* tokens, not guess. Libraries like `tiktoken` do this; most of the time the `usage` field on each response (0.1) is enough to see the trend.
::::

::::{dropdown} 🔧 Under the hood: prompt caching — why resending history is cheaper than it looks
:color: info

The naive column above overstates the *price* (not the size) of resending history, because providers cache prompts. If the first part of your request is **byte-identical** to a recent one, the provider skips recomputing it and bills those tokens at a fraction of the price (on Anthropic's API, cached input costs ~10× less; OpenAI discounts automatically).

The catch is that one word: *identical*. Caching works on a **stable prefix**:

- **Naive memory is append-only** → the entire previous prompt is the prefix of the next one → caches beautifully.
- **Window memory slides** → the front of the message list changes every turn → cache misses.
- **Summary memory mutates the system prompt** on every fold → busts the cache at position zero, the worst place.

So the strategies rank *differently* on size and on cache-friendliness — a real design tension. Production systems resolve it by compacting **rarely and in big jumps** instead of every turn (that's what Claude Code's compaction does), keeping the prefix stable between compactions. We come back to this with numbers in **0.8 (context engineering)**.
::::

```{note}
Everything in this notebook is **short-term memory**: the state of *one* conversation, gone when the process ends. "The assistant remembers I'm vegetarian *across sessions*" is a different problem — long-term memory, stored outside the message list and injected on demand. That's **2.8**.
```

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Tune the window.** Set `keep_turns=5` in `WindowMemory` and re-ask about the colour. Bigger window, more memory, more tokens per call — the whole trade-off in one number.
2. **Inspect the messages.** Print `m.messages()` after a few turns of `SummaryMemory` to see exactly what gets sent: system + summary + recent.
3. **Break the summarizer.** Change the summary instruction to "be extremely brief" and watch which facts survive and which get dropped. Summarization is lossy — you choose the losses.
::::

::::{dropdown} 🛠️ Solutions
:color: secondary

**1 — Tune the window:** with `keep_turns=5` all four facts fit, so the colour question succeeds — and the measurement table's `window` column rises accordingly. One number trades memory for tokens.

**2 — Inspect the messages:**

```python
for msg in summ.messages():
    print(f"[{msg['role']}]")
    print(msg['content'][:300], "\n")
```

**Done when:** you can point at the three layers in the output: system prompt, the folded summary riding inside it, and the verbatim recent turns.

**3 — Break the summarizer:** with "be extremely brief" the summary typically keeps the *latest* or most-repeated facts and silently drops the rest — often the dog's name or the sister. There's no error, no warning: the fact is just gone. That silence is the point: **summarization failures are invisible until someone asks the wrong question.** In production you'd protect the facts that matter with structured extraction (2.8) instead of trusting prose compression.
::::

::::{dropdown} 🔧 Common issues
:color: info

| Symptom | Likely cause | Fix |
|---|---|---|
| The summary silently drops facts | The summarizer keeps what looks salient, not what you need | Enumerate the non-negotiables in its instructions ("keep ALL facts about the user"); see also 0.8 |
| **429** errors during the measurement cell | 18 calls in a quick burst on the free tier | Re-run after a minute, shorten `TURNS`, or use a paid model |
| Provider error about the first message role | A window slice that opens with an `assistant` message | Keep the mid-pair guard in `WindowMemory.messages()` — that's what it's for |
| The window demo still recalls the colour | The model echoed the fact into a *later* reply, which is still inside the window | Inspect the printed window; echoes are memory too. Add filler turns to push everything out |
::::

You've now built every moving part by hand: model calls, structured output, the agent loop, workflows, reflection, human-in-the-loop, and memory.

**What's next:** in **0.7** we put all of it to work at once and build the thing this block has been quietly building toward — a real **coding agent**, the same species as Claude Code, Codex and OpenCode: file tools, a shell, a permission gate, and a verification loop. Then **0.8** adds the discipline that keeps long agent sessions alive: **context engineering**.